In [2]:
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Pose
from sensor_msgs.msg import JointState

from pymoveit2 import MoveIt2
from tf2_ros import Buffer, TransformListener

import json
import time

In [3]:
rclpy.init()

node = Node("moveit_jupyter_node")

print("ROS initialized")

ROS initialized


In [4]:
import threading
import time

def spin_some(duration=2.0):
    """    
    important: take some time to process incoming data!
    """
    import time
    end = time.time() + duration
    while time.time() < end:
        rclpy.spin_once(node, timeout_sec=0.1)
        
def spin_thread():
    """
    Background spinning to update incoming data
    """
    while rclpy.ok():
        rclpy.spin_once(node, timeout_sec=0.01)
        time.sleep(0.005)  # reduces CPU spikes

In [5]:
# if "spin_thread_started" not in globals():
#     spin_thread_started = True

#     import threading

#     def spin_thread():
#         while rclpy.ok():
#             rclpy.spin_once(node, timeout_sec=0.01)

#     threading.Thread(target=spin_thread, daemon=True).start()
#     print("Spin thread started")
# else:
#     print("Spin thread already running")

# setup

In [6]:
from pymoveit2 import MoveIt2
from tf2_ros import Buffer, TransformListener

from geometry_msgs.msg import Pose

tf_buffer = Buffer()
tf_listener = TransformListener(tf_buffer, node)

spin_some() # give it some time to update

In [7]:
moveit2 = MoveIt2(
    node=node,
    joint_names=[
        "joint_1","joint_2","joint_3",
        "joint_4","joint_5","joint_6",
    ],
    base_link_name="base_link",
    end_effector_name="tool_frame",
    group_name="arm",
)

# gripper

In [8]:
from rclpy.action import ActionClient
from control_msgs.action import GripperCommand

gripper_client = ActionClient(
    node,
    GripperCommand,
    "/gen3_lite_2f_gripper_controller/gripper_cmd"
)

In [9]:
def control_gripper(position, effort=1.0):
    """
    position:
        0.0 = open
        ~0.8–1.0 = closed
    """

    goal_msg = GripperCommand.Goal()
    goal_msg.command.position = position
    goal_msg.command.max_effort = effort

    print(f"Sending gripper command: {position}")

    # wait for server
    gripper_client.wait_for_server()

    send_goal_future = gripper_client.send_goal_async(goal_msg)

    rclpy.spin_until_future_complete(node, send_goal_future)
    goal_handle = send_goal_future.result()

    if not goal_handle.accepted:
        print("Gripper goal rejected")
        return

    result_future = goal_handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)

    print("Gripper done")

def open_gripper():
    control_gripper(0.0)

def close_gripper():
    control_gripper(0.8)  # adjust if needed

In [ ]:
close_gripper()
time.sleep(1)
open_gripper()

In [70]:
open_gripper()

Sending gripper command: 0.0
Gripper done


# get pose

In [10]:
def get_pose():
    spin_some(1.0)
    try:
        trans = tf_buffer.lookup_transform(
            "base_link",
            "tool_frame",
            rclpy.time.Time()
        )

        pose = Pose()
        pose.position.x = trans.transform.translation.x
        pose.position.y = trans.transform.translation.y
        pose.position.z = trans.transform.translation.z
        pose.orientation = trans.transform.rotation

        return pose
    except Exception as e:
        print("TF error:", e)
        return None
    
def print_pose(pose):
    print("Translation:",
          [pose.position.x, pose.position.y, pose.position.z])
    print("Quaternion:",
          [pose.orientation.x,
           pose.orientation.y,
           pose.orientation.z,
           pose.orientation.w])

test_pose0 = get_pose()
if test_pose0:
    print("Current pose:")
    print_pose(test_pose0)
else:
    print("failed to get pose.")
    

def make_pose(pos_xyz, orient_xyzw):
    pose = Pose()
    
    pose.position.x, pose.position.y, pose.position.z = pos_xyz
    pose.orientation.x, pose.orientation.y, pose.orientation.z, pose.orientation.w = orient_xyzw
    
    return pose

Current pose:
Translation: [0.5037739090109464, -0.22427072202545703, 0.11471652141402326]
Quaternion: [0.7306245025537038, -0.004027395951090627, 0.03243168020593197, 0.6819969226243078]


In [71]:
close_gripper()

Sending gripper command: 0.8
Gripper done


## save pose

In [11]:
grey_bottle_grasp_pose = make_pose(
    pos_xyz=(
        0.5037365889870258, 
        -0.2240841929310071, 
        0.11609242244597344
        ), 
    orient_xyzw=(
        0.7301995308282448, 
        -0.005421623908711407, 
        0.033587134403944244, 
        0.6823863682511068
        )
    )
print_pose(grey_bottle_grasp_pose)

Translation: [0.5037365889870258, -0.2240841929310071, 0.11609242244597344]
Quaternion: [0.7301995308282448, -0.005421623908711407, 0.033587134403944244, 0.6823863682511068]


In [12]:
# grey_bottle_pre_grasp_pose = get_pose()
# print_pose(grey_bottle_pre_grasp_pose)

In [13]:
grey_bottle_pre_grasp_pose =  make_pose(
    pos_xyz=[0.5035170876022704, -0.12519245872336004, 0.13016054581559786],
    orient_xyzw=[0.7122703498240701, 0.018009366124129053, 
                 0.044352911487721705, 0.700270969508137]
)
print_pose(grey_bottle_pre_grasp_pose)

Translation: [0.5035170876022704, -0.12519245872336004, 0.13016054581559786]
Quaternion: [0.7122703498240701, 0.018009366124129053, 0.044352911487721705, 0.700270969508137]


# get joint state

In [14]:
joint_state = None

def joint_callback(msg):
    global joint_state
    joint_state = msg

node.create_subscription(
    JointState,
    "/joint_states",
    joint_callback,
    10
)

spin_some(1)

print(joint_state)

sensor_msgs.msg.JointState(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1777568832, nanosec=699334362), frame_id='base_link'), name=['right_finger_bottom_joint', 'joint_1', 'joint_2', 'joint_4', 'joint_5', 'joint_3', 'joint_6'], position=[-0.019911933517456053, 0.03271080872111626, -2.085378704206623, 0.0670709131053186, 1.5838335655952704, -0.8429298997942123, -0.2751823049090163], velocity=[0.0, -7.137007583425824e-05, -2.1158659628816693e-05, -3.7090642116979656e-05, -0.00011102011344922246, 0.0, 3.434393621359027e-05], effort=[nan, 0.0068731061182916164, -14.017172813415527, 0.9950693845748901, 0.44595083594322205, 4.158229351043701, -0.3028472065925598])


# move pose

In [ ]:
pose = get_pose()

pose.position.z += 0.05  # move UP slightly

moveit2.move_to_pose(pose)

spin_some(5.0)

In [15]:
DEFAULT_SPEED = 0.3

import numpy as np

def reached_pose(target, pos_tol=0.01, ori_tol=0.05, check_orient=True):
    current = get_pose()

    # position
    p1 = np.array([
        current.position.x,
        current.position.y,
        current.position.z
    ])

    p2 = np.array([
        target.position.x,
        target.position.y,
        target.position.z
    ])

    pos_error = np.linalg.norm(p1 - p2)

    if not check_orient:
        return pos_error < pos_tol

    else:
        # orientation (quaternion distance)
        q1 = np.array([
            current.orientation.x,
            current.orientation.y,
            current.orientation.z,
            current.orientation.w
        ])

        q2 = np.array([
            target.orientation.x,
            target.orientation.y,
            target.orientation.z,
            target.orientation.w
        ])

        ori_error = np.linalg.norm(q1 - q2)

        return pos_error < pos_tol and ori_error < ori_tol

def move_to_pose(p, speed=None, wait=3.0):
    # Save current settings
    prev_vel = moveit2.max_velocity
    prev_acc = moveit2.max_acceleration

    # Apply new speed (or default)
    if speed is None:
        speed = DEFAULT_SPEED

    moveit2.max_velocity = speed
    moveit2.max_acceleration = speed

    # Execute motion
    moveit2.move_to_pose(p)

    time.sleep(wait)   # assuming background spin

    # Restore previous settings
    moveit2.max_velocity = prev_vel
    moveit2.max_acceleration = prev_acc

    return None

def move_to_pose_verified(p, speed=0.2, retries=2, wait=5.0, check_orient=True):
    for i in range(retries + 1):
        print(f"Attempt {i+1}")

        move_to_pose(p, speed, wait)

        time.sleep(1.0)  # let robot settle

        if reached_pose(p, check_orient):
            print("Reached target")
            return True

        print("Not reached, retrying...")

    print("FAILED to reach target")
    return False

In [16]:

move_to_pose_verified(grey_bottle_pre_grasp_pose, speed=0.2, wait=2, check_orient=False)


Attempt 1


[WARN] [1777568843.853954292] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777568843.854965197] [moveit_jupyter_node]: Joint states are available now


Not reached, retrying...
Attempt 2


[WARN] [1777568847.889356645] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777568847.890164586] [moveit_jupyter_node]: Joint states are available now


Not reached, retrying...
Attempt 3


[WARN] [1777568851.948968815] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777568851.949701116] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777568855.007343305] [moveit_jupyter_node]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.


Not reached, retrying...
FAILED to reach target


False

In [17]:
move_to_pose(grey_bottle_pre_grasp_pose, speed=0.2, wait=2)


[WARN] [1777568866.697042374] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777568866.697791229] [moveit_jupyter_node]: Joint states are available now


In [80]:
move_to_pose(grey_bottle_grasp_pose, speed=0.2, wait=2)

[WARN] [1777572790.010264970] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777572790.011037641] [moveit_jupyter_node]: Joint states are available now


In [ ]:
move_to_pose_verified(grey_bottle_grasp_pose, speed=0.1, wait=1, check_orient=False)

# behaviors

In [19]:

def lift(delta_z=0.08):
    pose = get_pose()
    pose.position.z += delta_z
    move_to_pose(pose, speed=0.3)

In [83]:
close_gripper()
lift(delta_z=0.20)

Sending gripper command: 0.8
Gripper done


[WARN] [1777573676.840912882] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777573676.841629609] [moveit_jupyter_node]: Joint states are available now


## shake water bottle

In [22]:
from tf_transformations import quaternion_from_euler, quaternion_multiply


def rotate_horizontal():
    pose = get_pose()

    current_q = [
        pose.orientation.x,
        pose.orientation.y,
        pose.orientation.z,
        pose.orientation.w
    ]

    # rotate around Y (pitch)
    delta_q = quaternion_from_euler(0, 1.57, 0)

    new_q = quaternion_multiply(delta_q, current_q)

    pose.orientation.x = new_q[0]
    pose.orientation.y = new_q[1]
    pose.orientation.z = new_q[2]
    pose.orientation.w = new_q[3]

    move_to_pose(pose, speed=0.2)

In [95]:
rotate_wrist(1.57/2, speed=2.0)

[WARN] [1777573934.361643608] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777573934.362208497] [moveit_jupyter_node]: Joint states are available now


In [88]:
import time
import numpy as np

ARM_JOINT_NAMES = [
    "joint_1",
    "joint_2",
    "joint_3",
    "joint_4",
    "joint_5",
    "joint_6",
]

def get_arm_joints():
    joint_dict = dict(zip(joint_state.name, joint_state.position))
    return [joint_dict[name] for name in ARM_JOINT_NAMES]

def oscillate_wrist_smooth(cycles=3, angle_deg=90, speed=2.0, dt=0.05):
    joint_idx = moveit2.joint_names.index("joint_6")

    base = get_arm_joints()
    base_angle = base[joint_idx]

    amp = np.deg2rad(angle_deg)

    # set speed once
    moveit2.max_velocity = speed
    moveit2.max_acceleration = speed

    steps_per_half = 10   # more steps = smoother

    for _ in range(cycles):
        # move forward (+45°)
        for i in range(steps_per_half):
            alpha = (i + 1) / steps_per_half
            joints = base.copy()
            joints[joint_idx] = base_angle + alpha * amp

            moveit2.move_to_configuration(joints)
            time.sleep(dt)

        # move back (to baseline)
        for i in range(steps_per_half):
            alpha = (i + 1) / steps_per_half
            joints = base.copy()
            joints[joint_idx] = base_angle + amp * (1 - alpha)

            moveit2.move_to_configuration(joints)
            time.sleep(dt)

In [90]:
oscillate_wrist_smooth(angle_deg=-90)

[WARN] [1777573824.539744273] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777573824.540390739] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777573824.546734073] [moveit_jupyter_node]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.
[WARN] [1777573824.548604115] [moveit_jupyter_node]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.
[WARN] [1777573824.552520551] [moveit_jupyter_node]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.
[WARN] [1777573824.608070666] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777573824.608781806] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777573824.615046777] [moveit_jupyter_node]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.
[WARN] [1777573824.678434863] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777573824.678985015] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777573824.74

In [23]:
def shake(axis='x', amplitude=0.04, cycles=8, speed=0.8):
    base_pose = get_pose()

    for i in range(cycles):
        pose1 = get_pose()
        pose2 = get_pose()

        if axis == 'x':
            pose1.position.x += amplitude
            pose2.position.x -= amplitude
        elif axis == 'y':
            pose1.position.y += amplitude
            pose2.position.y -= amplitude

        move_to_pose(pose1, speed)
        move_to_pose(pose2, speed)

In [ ]:
move_to_pose()

In [42]:
def get_arm_joints():
    joint_dict = dict(zip(joint_state.name, joint_state.position))
    return [joint_dict[name] for name in moveit2.joint_names]

def rotate_wrist(delta, speed=0.5):
    joints = get_arm_joints()

    joints[-1] += delta   # safer than index 5

    moveit2.max_velocity = speed
    moveit2.max_acceleration = speed

    moveit2.move_to_configuration(joints)
    spin_some(2.0)


rotate_wrist(-1.57*3, speed=2.0)
rotate_wrist(1.57*3, speed=2.0)

[WARN] [1777569475.210586364] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777569475.211379011] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777569475.235729102] [moveit_jupyter_node]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.
[WARN] [1777569477.231284036] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777569477.232264421] [moveit_jupyter_node]: Joint states are available now


In [79]:
for i in range(3):
    rotate_wrist(-1.57*2, speed=2.0)
    rotate_wrist(1.57*2, speed=2.0)

[WARN] [1777572726.155831266] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777572726.156383443] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777572726.180823474] [moveit_jupyter_node]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.
[WARN] [1777572728.176162706] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777572728.177114594] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777572730.192340385] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777572730.193107538] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777572732.212190189] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777572732.212954200] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777572733.769885957] [moveit_jupyter_node]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.
[WARN] [1777572734.230151932] [moveit_jupyter_node]: Joint states are not avai

In [76]:
rotate_wrist(1.57*2, speed=2.0)

[WARN] [1777572638.657331743] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777572638.657966474] [moveit_jupyter_node]: Joint states are available now


In [30]:
print(joint_state.name)
print(len(joint_state.position))

['right_finger_bottom_joint', 'joint_1', 'joint_2', 'joint_4', 'joint_5', 'joint_3', 'joint_6']
7


In [ ]:
def shake_joint(joint_idx=4, amp=0.5, cycles=8):
    joints = get_arm_joints()

    for _ in range(cycles):
        joints[joint_idx] += amp
        moveit2.move_to_configuration(joints)
        time.sleep(0.2)

        joints[joint_idx] -= 2 * amp
        moveit2.move_to_configuration(joints)
        time.sleep(0.2)

        joints[joint_idx] += amp

In [48]:
print(moveit2.max_velocity)
print(moveit2.max_acceleration)

2.0
2.0


In [26]:
rotate_horizontal()

[WARN] [1777569058.155411580] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777569058.156148003] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777569058.679106549] [moveit_jupyter_node]: Planning failed! Error code: FAILURE
[WARN] [1777569058.679852191] [moveit_jupyter_node]: Cannot execute motion because the provided/planned trajectory is invalid.


In [24]:
shake()

[WARN] [1777568985.104653448] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777568985.105354670] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777568985.622256092] [moveit_jupyter_node]: Planning failed! Error code: FAILURE
[WARN] [1777568985.622958921] [moveit_jupyter_node]: Cannot execute motion because the provided/planned trajectory is invalid.
[WARN] [1777568988.626440540] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777568988.627308896] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777568993.660356138] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777568993.661093260] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777568996.693503659] [moveit_jupyter_node]: Joint states are not available yet!
[INFO] [1777568996.694368594] [moveit_jupyter_node]: Joint states are available now
[WARN] [1777568999.731863206] [moveit_jupyter_node]: Action 'execute_trajectory' was unsucc

KeyboardInterrupt: 

## tilt cup

# shut down

In [ ]:
node.destroy_node()
rclpy.shutdown()

print("Shutdown complete")